In [1]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch
from torch import nn
import pandas as pd
from sentence_transformers import InputExample

In [2]:
# Base embedding model
base_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')


In [5]:
df = pd.read_csv("final_dataframe.csv")

In [7]:
df.iloc[0,0]

'Abhishek Chaudhary \n   ac5712916@gmail.com  |  +91-9781673453  |  Punjab, India  \n                                                        \n \nSkills \n        Python  |  kotlin  |  OOPS   | HTML| CSS  | Critical Thinking \n \nProjects \nWeather Prediction App \n                       Python \n• \nIt is used to predict the weather. \n• \nIt can give weather reports manually. \n• \nIt also display’s the current day weather. \n• \nIn this application, API is used. \nCar Karo App      \n   HTML, CSS, JavaScript  \n• \nIt is car booking application where one can book the cab at reasonable prices.  \n• \nSharing and user-friendly application. \n• \nLocation API is used to track the cab. \n• \nPayment method can be both offline and online. \n \n \nCertifications  \n• \nPython Certificate – Nice computers \n \nAchievements  \n• \nDistrict Cricket player \n• \nState karate player. \nEducation  \nD.A.V. Institute of Engineering and Technology                                                  

In [4]:
df.head()

,resume,jd,new_ats
0,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nPosition: Junior PHP Dev...,75
1,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Summary:\n\nWe are s...,65
2,Abhishek Chaudhary \n ac5712916@gmail.com |...,Job Title: Sr. Website UI/UX Designer and Deve...,65
3,Abhishek Chaudhary \n ac5712916@gmail.com |...,We are looking for a passionate and enthusiast...,85
4,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Title: Software Engi...,65


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

class ResumeDataset(Dataset):
    def __init__(self, df, base_model):
        self.resumes = df['resume'].tolist()
        self.jds = df['jd'].tolist()
        self.scores = df['new_ats'].astype(float).tolist()
        self.base_model = base_model

    def __len__(self):
        return len(self.resumes)

    def __getitem__(self, idx):
        emb1 = self.base_model.encode(self.resumes[idx], convert_to_tensor=True)
        emb2 = self.base_model.encode(self.jds[idx], convert_to_tensor=True)
        score = torch.tensor(self.scores[idx], dtype=torch.float32)
        return torch.cat([emb1, emb2]), score

#dataloader
dataset = ResumeDataset(df, base_model)
train_dataloader = DataLoader(dataset, shuffle=True, batch_size=16)


#regression model
import torch.nn as nn

class ATSRegressor(nn.Module):
    def __init__(self, embedding_dim, hidden_size=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim*2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.fc(x)


#training loop

model = ATSRegressor(embedding_dim=base_model.get_sentence_embedding_dimension())
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

for epoch in range(3):
    total_loss = 0
    for x, y in train_dataloader:
        preds = model(x)
        loss = criterion(preds.squeeze(), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_dataloader)}")


Epoch 1, Loss: 4671.879032019413
Epoch 2, Loss: 4634.975442905618
Epoch 3, Loss: 4590.532431246054


In [24]:
torch.save(model.state_dict(), "ats_regression_model.pt")
